# Pytorch? nn.Embedding
- Pytorch? Embedding Layer? word2vec? ????? word embedding vector? ?? **Lookup Table**??.
    - ??? **??? ?? index**? ???? ???? Embedding Layer? **? index? Vector**? ????.
    - ??? ???? ?? ??? ??? ??? ?? ??? Embedding Layer? vector?? ???? ??.
    - Word2Vec? embedding vector ??? nn.Embedding? ??? ??? ??? ?? ?? ???? ??? ???? ??.

In [ ]:
import torch
import torch.nn as nn

embed = nn.Embedding(
    num_embeddings=20_000,  # vocab size(????? ???.) -> ? ??? ??? ?? embedding vector? ???.
    embedding_dim=200,      # embedding vector? ??? -> ?? ??? ??? ??(feature)? ????.
)

In [ ]:
# embed.weight
embed.weight.shape

In [ ]:
# embedding layer? ?? - ??? ???? ???? ID(**??**)? 1?? ??? ??.

# doc = ??-30 | ??-159 | ??-9000 | ???-326   3?? ??, 4?? ??? ??
doc = torch.tensor([[30, 159, 9000, 326], [30, 159, 9000, 326], [30, 159, 9000, 326]], dtype=torch.int64) 
embedding_vector = embed(doc)
embedding_vector.shape

#[1: batch_size, 4:seq_len, 200: embedding vector ???]

In [ ]:
embedding_vector[0, 0]   #??? ??? ??? ??? ???? 200?? ?? ??

In [ ]:
embed.weight[30] # = embedding_vector[0,0] # weight? embedding vector???. 
# ?? doc?? ??? 30??? ???? ?? ??? ?? ?? ?? ?? ?

# ??? ?? ?? ????(Sentiment Analysis)

## ????(Sentiment Analysis) ??
??? ???? **???? ?**?? **????**?? ?? **????** ??? ???? ?? ??(??) ????? ??.   
?? ?? ??? ??? ???? ?? ?? ??? ?? ?? ??? ??? ??? ????.

# Dataset, DataLoader ??

## Korpora?? Naver ?? ?? dataset ????
- https://ko-nlp.github.io/Korpora/ko-docs/corpuslist/nsmc.html
- http://github.com/e9t/nsmc/
    - input: ????
    - output: 0(?????), 1(?????)
### API
- **corpus ????**
    - `Korpora.load('nsmc')`
- **text/label ??**
    - `corpus.get_all_texts()` : ?? corpus? text?? tuple? ??
    - `corpus.get_all_labels()`: ?? corpus? label?? list? ??
- **train/test set ??? ??**
    - `corpus.train`
    - `corpus.test`
    - `LabeledSentenceKorpusData` ??? text? label?? ??? ??.
        - `LabeledSentenceKorpusData.texts`: text? tuple? ??.
        - `LabeledSentenceKorpusData.labels`: label? list? ??.

## ??? ??

In [ ]:
import os
import time

from Korpora import Korpora

corpus = Korpora.load("nsmc")   #cors corpus.train, corpus.test?? ?? ?? ???? ??? ????.

In [ ]:
all_inputs = corpus.get_all_texts()  # inputs: ??? ??
all_labels = corpus.get_all_labels() # outputs: labels ?? - 0: ??, 1: ??

In [ ]:
all_inputs[:5]

In [ ]:
all_labels[:5]

In [ ]:
len(all_inputs)

In [ ]:
corpus.train

In [ ]:
corpus.train.texts

In [ ]:
corpus.test

In [ ]:
corpus.test.texts[:5]

In [ ]:
corpus.test.labels[:5]

## ??? (??? ??)
1. ??? ?? token?(??)? ?? ??.
    - konlpy? token? ? ? ?? ? ???? ???.
2. 1?? ??? corpus? BPE ? token?
   
### ??? ??

#### ??? ?? ??

In [ ]:
import string
list(string.punctuation)

In [ ]:
# ??? = cleansing + ???(normalize)
from konlpy.tag import Okt
import string
import re

okt = Okt()
def text_preprocessing(text):
    """
    1. ?? -> ???? ?? # ???? ??? ??? ??? ??!
    2. ??? ??
    3. ??? ?? ???
    4. ???? ??? ? ? ?? ??? ???? ??? ??.
    """
    # ????
    text = text.lower()   
    # ??? ?? (stop word(???))
    text = re.sub(f"[{string.punctuation}]", " ", text)
    # ???
    tokens = okt.morphs(text, stem=True) # ??? ?? # stem: ????. norm=True (??? ??)
    return ' '.join(tokens) # ["??", "??", ..] -> str "?? ?? ??" # ???? ?? ???? ???
    

In [ ]:
print(all_inputs[101])
text_preprocessing(all_inputs[101])


In [ ]:
s = time.time()
# train set ???
train_texts = corpus.train.texts
train_inputs = [text_preprocessing(txt) for txt in train_texts]
train_labels = corpus.train.labels

# test set ???
test_texts = corpus.test.texts
test_inputs = [text_preprocessing(txt) for txt in test_texts]
test_labels = corpus.test.labels
e = time.time()

print('??? ????(?):', e-s) 

In [ ]:
import pickle
import os
# ???? ??? ??
os.makedirs("datasets/nsmc", exist_ok=True)
with open("datasets/nsmc/preprocessing_trainset.pkl", "wb") as fw:
    pickle.dump({"input":train_inputs, "output": train_labels}, fw)

In [ ]:
with open("datasets/nsmc/preprocessing_testset.pkl", "wb") as fw:
    pickle.dump({"input": test_inputs, "output":test_labels}, fw)

In [ ]:
# pickle? ??? ???? ???? ?? ??.
import pickle
with open("datasets/nsmc/preprocessing_trainset.pkl", "rb") as fr:
    train_dict = pickle.load(fr)

with open("datasets/nsmc/preprocessing_testset.pkl", "rb") as fr:
    test_dict = pickle.load(fr)

In [ ]:
train_inputs = train_dict['input']
train_labels = train_dict['output']

test_inputs = test_dict['input']
test_labels = test_dict['output']


In [ ]:
all_inputs = train_inputs + test_inputs # vocab ??? ??.

In [ ]:
len(train_inputs), len(test_inputs), len(all_inputs)

### ???
- Subword ?? ??? ??
- Byte Pair Encoding ???? huggingface tokenizer ??
    - BPE: ??? ?? ??? ??? ?? ?? ???? ?? ?(byte paire)? ?? ??? ????? ????.
    - https://huggingface.co/docs/tokenizers/quicktour
    - `pip install tokenizers`

?? ????????<br>
??? ??? ??? ?? ?? ??? ???, ? ??? ?? ??? ???? ?????.

???? ????(Subword) ??? **BPE(Byte Pair Encoding)**? ?????.

?? BPE? ??
- ??? ?? ??? ?? ?, ?? ???? ???? ??? ??? ????<br>

- ??? ??? ???? ?? ??? ??? ?? ??

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE #, Unigram, WordPiece
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

vocab_size = 30_000 # ?? ????? ?? ???
min_frequency = 5   # ??? ??? ?? ?? ?.

tokenizer = Tokenizer(
    BPE(unk_token="[UNK]")
)
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=min_frequency,
    special_tokens=["[PAD]", "[UNK]"],
    continuing_subword_prefix="##" # ??? ??? ??? subword? ?? ?? ##? ???.
    #"????" -> "??", "??" => "??", "##??"
)
# ??
tokenizer.train_from_iterator(all_inputs, trainer=trainer) # vocab ?? == tokenizer??.

In [ ]:
tokenizer.token_to_id("[PAD]")

In [ ]:
# ? vocab size:
tokenizer.get_vocab_size()

In [ ]:
# ??
os.makedirs("saved_models/nsmc", exist_ok=True)
tokenizer.save("saved_models/nsmc/tokenizer_bpe.json")

# load_tokenizer = Tokenizer.from_file("saved_models/nsmc/tokenizer_bpe.json")

In [ ]:
idx = 11290
print(all_inputs[idx])
tokens = tokenizer.encode(all_inputs[idx])  # ??? ??(subword) ??? ??.
print(tokens.ids)     # ??? ???? ?? ?? ID? ??
print(tokens.tokens)   

In [ ]:
e = tokenizer.encode("pytorch? pandas? numpy? python ????? ???.")
e.tokens

In [ ]:
tokenizer.decode(e.ids)

In [ ]:
tokenizer.decode(tokens.ids)

## Dataset, DataLoader ??

In [ ]:
# dset[0]
tokenizer.encode(train_inputs[0]).ids, train_labels[0]

In [ ]:
# Pytorch ??? ?? Dataset(Custom Dataset) ??
# 1. Dataset ??
# 2. __len__(self) : ? ???? ?? ??
# 3. __getitem__(self, index) : index? x,y? ??. - ??[index]

In [ ]:
train_inputs

In [ ]:
train_labels
[tokenizer.encode(txt).ids for txt in train_inputs[:4]]

In [ ]:
tokenizer.token_to_id

In [ ]:
tokenizer.token_to_id("[PAD]")

In [ ]:
for txt in train_inputs:
    print(tokenizer.encode(txt))
    break

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class NSMCDataset(Dataset):
    def __init__(self, texts, labels, max_length, tokenizer):
        """
        texts: list - ?? ??. ???? ???? ??? ???. ["??", "??", ...]
        labels: list - ?? Label(?/??) ??. 
        max_length: ?? ??? ?? token ??. ?? ??? ???? max_length? ???.
        tokenizer: Tokenizer
        """
        self.max_length = max_length
        self.tokenizer = tokenizer
        self.labels = labels
        # self.texxts: ???? - token id? ??? ??(??). ???? max_length? ??.
        #              max_length  ?? ??? [PAD] ??, max_length?? ??? ????.
        self.texts = [self.__pad_token_sequences(tokenizer.encode(txt).ids) for txt in texts]

    ###########################################################################################
    # id? ??? ?? ?? token list? ??? ?? ?? [20, 2, 1] => [20, 2, 1, 0, 0, 0, ..]
    # max_length? ?? ???? ??? ???? ??.
    ############################################################################################
    def __pad_token_sequences(self, token_sequences): # ?? __ ??? class???? ?????.???? ?? ??. (JAVA??? private)
        """
        id? ??? ?? ??(??)? token_id list? ??? max_length ??? ??? ???
        max_length ?? ???? ??? [PAD] ??, ??? max_length ??? ???.
            ex) max_length = 5, [PAD] token id? 0
            - [20, 2, 1] => [20, 2, 1, 0, 0]
            - [20, 30, 40, 50, 60, 70, 80] -> [20, 30, 40, 50, 60]
        """
        pad_token_id = self.tokenizer.token_to_id("[PAD]")
        seq_len = len(token_sequences) # ???? ?? ??.
        result = None
        if seq_len > self.max_length: #????
            result = token_sequences[:self.max_length]
        else:
            result = token_sequences + ([pad_token_id] * (self.max_length - seq_len))
        
        return result
        
    def __len__(self):
        return len(self.labels) # ? ??? ??? ??.

    def __getitem__(self, idx):
        """
        idx ?? text? label? ?? ??? type?? ???? ??
        Parameter
            idx: int ??? index
        Return
            tuple: (torch.LongTensor, torch.FloatTensor) - ?? ??_id ???, ?? Label
        """
        txt = self.texts[idx]
        label = self.labels[idx]

        return (torch.tensor(txt, dtype=torch.int64), torch.tensor([label], dtype=torch.float32))
    

In [ ]:
# ?? ??? ???? ??.
all_input_length = [ len(tokenizer.encode(txt)) for txt in all_inputs]
all_input_length[:5]

In [ ]:
import numpy as np
np.min(all_input_length), np.max(all_input_length)

In [ ]:
np.quantile(all_input_length, q=[0.9, 0.95])
# ?? ? 90%? ???? 29? ??, 95%? 41? ??.

In [ ]:
200000 * 0.9

In [ ]:
MAX_LENGTH = 30
trainset = NSMCDataset(train_inputs, train_labels, MAX_LENGTH, tokenizer)
testset = NSMCDataset(test_inputs, test_labels, MAX_LENGTH, tokenizer)

In [ ]:
len(trainset), len(testset)

In [ ]:
# trainset[0]
trainset[10]

In [ ]:
BATCH_SIZE = 64
train_loader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
test_loader = DataLoader(testset, batch_size=BATCH_SIZE)

In [ ]:
len(train_loader), len(test_loader)

In [ ]:
import torch
import torch.nn as nn

# ???? ??? : 10?
# ??? ?? ??? : 3??

e_layer = nn.Embedding(
    num_embeddings=10,
    embedding_dim=3,
)
# 10 X 3 ??? ?? -> weight ??? ?? ???? embedding vector?

In [ ]:
e_layer.weight

In [ ]:
sent = "?? ?? ??" # ["??", "??", "??"]
# token = tokenizer.encode(sent).ids
token= torch.tensor([0, 4, 2], dtype=torch.int64) # .ids? ???? ????? ??? ??? ??.
e_layer(token)
'''
[
[-0.4482, -0.5812, -1.2675], <- 0 :??
[-1.0349, -0.9678,  0.9191], <- 4 :??
[-0.1923, -0.7232, -0.0848]  <- 2 :??
]
'''

# ???
- Embedding Layer? ??? Word Embedding Vector? ????.
- LSTM? ??? Feature ??
- Linear + Sigmoid? ?? ??? ?? ??
  
![outline](figures/rnn/RNN_outline.png)

- ?? ??? ?? ?? ???? ?? ????? ??? hidden state?? ???.

## ?? ??

In [ ]:
!pip install torchinfo

In [ ]:
import torch
import torch.nn as nn
from torchinfo import summary # ?? ??(summary) ?? ?????
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
# ????
class NSMCClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers, bidirectional=True, dropout_rate=0.2):
        """
        Args:
            vocab_size(int) - ????? ? ???
            embedding_dim - embedding vector? ???
            hidden_size(int) - LSTM? hidden state? featrue?
            num_layers(int) - LSTM? layers? ??
            bidirectional(bool) - LSTM? ??? ??
            dropout_rate(float) - LSTM? ?? ??? layer? ??? ?? ??? dropout ??. Dropout Layer? dropout ??.
        """
        super().__init__()
        # ??? ??? Layer?? ?? - Embedding, LSTM, Dropout, Linear(???). Sigmoid
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,   # ? ??(??)? -> tokenizer? ??? ? ???.
            embedding_dim=embedding_dim, # ??? ??? ???.
            padding_idx=0                # [PAD]? ?? ID(tokenizer.token_to_index("[PAD]"))
                                         # ??? ???? padding??? ???? ???. ???? ??? ??? ??? ??
        )
        # embedding layer? ?? shape: (batch_size:64, seq_length: ?? ???, embedding_dim(embedding ???))

        self.lstm = nn.LSTM(
            input_size=embedding_dim,  # ?? ??(??)? feature ? (embedding layer -> LSTM?? ???.)
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional, 
            dropout=dropout_rate if num_layers > 1 else 0, # stacked rnn? ?? ??.
        )

        self.dropout = nn.Dropout(dropout_rate) # LSTM? Linear ??? ??? ??? ??? ??.

        # lstm? ?? :  out, (hidden, cell)
        # out : ?? ????? hidden state? - [seq_len, batch, hidden_size * bidirectional??(0,1)]
        # hidden : ??? ????? hidden state? (?????) -[bidirectinal * num_layer(??), batch, hidden]
        # cell : ??? ????? cell state? (?????)

        input_features = hidden_size*2 if bidirectional else hidden_size
        self.classifier = nn.Linear(input_features, 1) # ?? 1 : ???? -> positive? ??.
        self.sigmoid = nn.Sigmoid()        # classifier? ???? ??(0 ~ 1) ??? ???? ??.

    def forward(self, X):
        """
        Args
            X(tensor): ?? ?? ?? ???. shape: [batch_size, seq_length(max_length : ?? ?? ???)] - [64, 30]
        """
        embedding_vectors = self.embedding(X)
        # [batch, seq_len] -> embedding -> [batch, seq_len, embedding_dim (embedding_???)]
        # LSTM - batch_first = False : ?? shape - [seq_len, batch_size, embedding_dim]
        # embedding_vector? batch ?? seq_len ?(?? ?? = ?? index)? ????.
        embedding_vectors = embedding_vectors.transpose(1, 0) # transpose() : ? ??? ?? ?? ??? ??? ? ??.
        out, _ = self.lstm(embedding_vectors)
        # out.shape: [seq_len, batch, hidden_size * (2 if bidirectional else 1)]
        # classifier(linear)?? out? ??? index(??? seq) ?? ??.
        output = self.dropout(out[-1])
        output = self.classifier(output)
        
        last_output = self.sigmoid(output)
        return last_output

In [ ]:
# ev.transpose(1, 0)
# # ????? ?? : index? ???? ? ??
# # ???? ? : ??? ?? index? ? ??
# # 1 -> 0, 0 -> 1

# ev = [
#     [10, 20, 30],
#     [40, 50, 60]
# ] # shape: (2, 3)
# a = ev.transpose(1, 0)
# 10 idx : [0, 0] -> [0, 0] # ??? 0, 0
# 20 idx : [0, 1] -> [1, 0]
# 30 idx : [0, 2] -> [2, 0]
# 40 idx : [1, 0] -> [0, 1]

# a[
#     [10, 40],
#     [20, 50],
#     [30, 60],
# ] # transpose ??? shape? (3, 2)? ??. transpose ->  ?? ??? ??? ?. reshape? ??? ??? ??.

# b = a.reshape(3, 2)
# [ 
#     [10, 20],
#     [30, 40],
#     [50, 60]
# ]

In [ ]:
# [20, 50, 8, 0, 0] -> embedding layer ->[[0.1,0.2,,0.3], [1.2, 0.2, -1.2], [], [], []]  (1,5,3)

In [ ]:
# tokenizer.encode("? ?? ?? ????.").ids

## ?? ??

In [ ]:
VOCAB_SIZE = tokenizer.get_vocab_size() # ? ???
EMBEDDING_DIM = 100
HIDDEN_SIZE = 64
NUM_LAYERS = 2
BIDIRECTIONAL=True # ???
DROPOUT_RATE = 0.3

# ??? ???? ???. => EMBEDDING_DIM, HIDDEN_SIZE, NUM_LAYERS? ?? ??.
# Auto regressive ??? ??? => BIDIRECTIONAL=True(???)

model = NSMCClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    bidirectional=BIDIRECTIONAL,
    dropout_rate=DROPOUT_RATE
)
model = model.to(device)
print(model)

In [ ]:
# summary? ?? ??
i = torch.randint(1, 10, (64, MAX_LENGTH)) # int 64 ??? dummy ?? ???. 
# ?? shape : (batch, seq_len)
summary(model, input_data=i, device=device)
# summary(??, input_shape) => ????? ??????(float32)? ???? ??.

## ??

### Train/Test ?? ??

In [ ]:
# 1 epoch train ?? ??.
def train(model, dataloader, loss_fn, optimizer, device="cpu"):
    # 1. ??? train ?? ??? ??.
    model.train()
    # 2. ??? device? ??
    model = model.to(device)
    total_loss = 0.0 # step? loss? ??.
    # step ??? ?? ?? ( batch )
    for X, y in dataloader:
        # 1. X, y? device? ??
        X, y = X.to(device), y.to(device)
        # 2. ??
        pred = model(X)
        # 3. loss ??
        loss = loss_fn(pred, y) # ????? ???
        # 4. gradient ??
        loss.backward()
        # 5. ???? ???? : w.data - w.grad * lr
        optimizer.step()
        # 6. gradient ???
        optimizer.zero_grad()
        # 7. loss ??
        total_loss += loss.item()
    # 1 epoch ?? ??
    return total_loss / len(dataloader) # 1 epoch? train loss? ??. (total loss / step??)

In [ ]:
# 1 epoch ??/?? ??
def test(model, dataloader, loss_fn, device="cpu"):
    # 1. ??? eval ??? ??, model? device? ??
    model.eval()
    model = model.to(device)
    # loss, accuracy ??
    total_loss = 0.0
    total_acc = 0.0
    with torch.no_grad(): # ??? ? gradient ??? ??? ???.
        for X, y in dataloader:
            # 1. step ??.
            # 1. X, y? device? ??
            X, y = X.to(device), y.to(device)
          
            # 2. ??
            pred_proba = model(X) # ??? ??
            pred_label = (pred_proba > 0.5).type(torch.int32) # True, False? 1,0?? ???? ??.
            total_loss += loss_fn(pred_proba, y).item()
            total_acc += (pred_label == y).sum().item() # T, F? ??? ??? int??? ??.
        
        # return loss, acc
        return total_loss / len(dataloader), total_acc / len(dataloader.dataset)
    # pred_proba = 1 if [0.7, 0.2, 0.6] > thresh else 0

### Train

In [ ]:
LR = 0.0001
EPOCHS = 3
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [ ]:
import time

s = time.time()
train_loss_list = []
val_loss_list = []
val_acc_list = []

for epoch in range(EPOCHS):
    train_loss = train(model, train_loader, loss_fn, optimizer, device)
    val_loss, val_acc = test(model, test_loader, loss_fn, device)
    train_loss_list.append(train_loss)
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)
    print(f"[{epoch}/{EPOCHS}] train loss: {train_loss}, val loss: {val_loss}, val acc: {val_acc}")
e = time.time()
print(e-s)

In [ ]:
# next(iter(train_loader))[1].shape

<!-- ## ???? -->

## ?? ??

In [ ]:
torch.save(model, 'saved_models/nsmc/model.pt')

In [ ]:
#????
load_model = torch.load('saved_models/nsmc/model.pt', weights_only=False)

# ???

## ??? ???

In [ ]:
from konlpy.tag import Okt

morph_tokenizer = Okt()

def text_preprocessing(text):
    
    text = text.lower()
    text = re.sub(f"[{string.punctuation}]+", ' ', text)
    return ' '.join(morph_tokenizer.morphs(text, stem=True)) # stem = True : normalization

In [ ]:
def pad_token_sequences(token_sequences, max_length):
    """padding ?? ???."""
    pad_token = tokenizer.token_to_id('[PAD]')  
    seq_length = len(token_sequences)           
    result = None
    if seq_length > max_length:                 
        result = token_sequences[:max_length]
    else:                                            
        result = token_sequences + ([pad_token] * (max_length - seq_length))
    return result

In [ ]:
def predict_data_preprocessing(text_list):
    """
    ??? ??? ??? input data? ??
    Parameter:
        text_list: list - ??? ?????
    Return
        torch.LongTensor - ?? token_id tensor
    """
   
    #cleansing + ???
    text_list = [text_preprocessing(txt) for txt in text_list]
    # text -> ???
    token_list = [tokenizer.encode(txt).ids for txt in text_list]
    # ?? ???? ??(size)?  max_length? ???
    token_list = [pad_token_sequences(token, MAX_LENGTH) for token in token_list]
    return torch.tensor(token_list, dtype=torch.int64)

## ??

In [ ]:
text_preprocessing("?? ???? ?????")

In [ ]:
comment_list = ["? ?? ????.", "?? ?? ??? ??", "?? ???? ?????", "?? ??? ?? ??? ????.", "?? ???? ?????", "?! ?? ??.", "?????.", "?? ? ??. ?? ? ? ??? ??.", "? ????"]
input_tensor = predict_data_preprocessing(comment_list)
input_tensor.shape

In [ ]:
input_tensor

In [ ]:
def predict(model, comment_list:list[str], input_tensor:torch.tensor, device='cpu'):
    """
    model? comment_list(?? id? ??? tensor)? ???? ??/???? ???? ??.
    ?? ??
        comment(??) label ??
        "? ????."  ??  0.9    (???? ??? ??)
        "???."       ?? 0.87   (???? ??? ???)
     """
    model.eval() # ????? ??
    model = model.to(device)
    with torch.no_grad():
        pred = model(input_tensor) # shape: (batch, 1) -> positive ? ??
        for txt, pos_proba in zip(comment_list, pred):
            label = "???" if pos_proba.item() > 0.5 else "???" # 0.5 -> threshold
            proba = pos_proba.item() if pos_proba.item() > 0.5 else 1-pos_proba.item() #??
            print(txt, label, round(proba, 3), sep='\t')


In [ ]:
predict(model, comment_list, input_tensor, device)

In [ ]:
print("????? ??? ?????. ????? 'quit'? ?????")
while True:
    comment = input("??:")
    if comment == "quit":
        print('??')
        break
    comment_list = [comment]
    input_tensor = predict_data_preprocessing([comment])
    predict(model, comment_list, input_tensor, device)